# IHD final fibroblast refinement

Input: `adata_celltype_corrected_v3.h5ad`, produced by `02_IHD_compartment_refinement_to_v3.ipynb`.

This notebook reclusters the fibroblast compartment, subclusters the recorded main clusters, applies the original marker-guided mappings, transfers the refined labels back to the full AnnData object, and saves `adata_celltype_final_corrected_v4.h5ad` with `celltype_final_corrected_v4`.

In [ ]:
from pathlib import Path
import scanpy as sc
import pandas as pd
import numpy as np

INPUT_FILE = Path("adata_celltype_corrected_v3.h5ad")
OUTPUT_FILE = Path("adata_celltype_final_corrected_v4.h5ad")
if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Required input not found: {INPUT_FILE.resolve()}")

adata = sc.read_h5ad(INPUT_FILE)
required_label = "celltype_final_corrected_v3"
if required_label not in adata.obs.columns:
    raise KeyError(f"Required annotation column {required_label!r} is missing.")


## Fibroblast compartment and marker review

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

def present(genes, adata):
    return [g for g in genes if g in adata.var_names]

# --- subset ---
fb_sub = adata[adata.obs["celltype_final_corrected_v3"] == "FB"].copy()
fb_sub

# --- preprocessing (Xenium-friendly) ---
sc.pp.normalize_total(fb_sub, target_sum=1e4)
sc.pp.log1p(fb_sub)

# --- clustering (NO HVG, NO scale) ---
sc.tl.pca(fb_sub, n_comps=30)
sc.pp.neighbors(fb_sub, n_neighbors=15, n_pcs=30)
sc.tl.leiden(fb_sub, resolution=1.6, flavor="leidenalg")
sc.tl.umap(fb_sub)

# --- marker sets ---
cm_markers = present(["TNNT2", "MYH6", "MYH7", "NPPA", "NPPB"], fb_sub)
fb_markers = present(["COL1A1", "DCN", "LUM", "PDGFRA"], fb_sub)
vasc_markers = present(["PECAM1", "VWF", "CD93"], fb_sub)
imm_markers = present(["PTPRC", "CD3D", "CD3E", "NKG7"], fb_sub)

print("CM:", cm_markers)
print("FB:", fb_markers)
print("Vascular:", vasc_markers)
print("Immune:", imm_markers)

# --- robust expression-based scoring ---
fb_sub.obs["CM_mean"] = np.array(fb_sub[:, cm_markers].X.mean(axis=1)).flatten()
fb_sub.obs["FB_mean"] = np.array(fb_sub[:, fb_markers].X.mean(axis=1)).flatten()
fb_sub.obs["Vasc_mean"] = np.array(fb_sub[:, vasc_markers].X.mean(axis=1)).flatten()
fb_sub.obs["Immune_mean"] = np.array(fb_sub[:, imm_markers].X.mean(axis=1)).flatten()

# --- optional scanpy scores ---
sc.tl.score_genes(fb_sub, cm_markers, score_name="CM_score")
sc.tl.score_genes(fb_sub, fb_markers, score_name="FB_score")
sc.tl.score_genes(fb_sub, vasc_markers, score_name="Vascular_score")
sc.tl.score_genes(fb_sub, imm_markers, score_name="Immune_score")

# --- derived metrics (FB-focused) ---
fb_sub.obs["FB_minus_CM"] = fb_sub.obs["FB_mean"] - fb_sub.obs["CM_mean"]
fb_sub.obs["FB_minus_Vasc"] = fb_sub.obs["FB_mean"] - fb_sub.obs["Vasc_mean"]
fb_sub.obs["FB_minus_Immune"] = fb_sub.obs["FB_mean"] - fb_sub.obs["Immune_mean"]

fb_sub.obs["CM_minus_FB"] = fb_sub.obs["CM_mean"] - fb_sub.obs["FB_mean"]
fb_sub.obs["Vasc_minus_FB"] = fb_sub.obs["Vasc_mean"] - fb_sub.obs["FB_mean"]
fb_sub.obs["Immune_minus_FB"] = fb_sub.obs["Immune_mean"] - fb_sub.obs["FB_mean"]

# --- cluster summary ---
fb_scores = (
    fb_sub.obs
    .groupby("leiden", observed=False)[[
        "CM_mean", "FB_mean", "Vasc_mean", "Immune_mean",
        "FB_minus_CM", "FB_minus_Vasc", "FB_minus_Immune"
    ]]
    .mean()
    .sort_values("FB_minus_CM", ascending=False)
)

print(fb_scores.round(3))

# --- visualization ---
sc.pl.umap(
    fb_sub,
    color=[
        "leiden",
        "FB_mean",
        "CM_mean",
        "Vasc_mean",
        "Immune_mean",
        "FB_minus_CM",
        "FB_minus_Vasc"
    ],
    vmax="p99"
)

In [ ]:
import scanpy as sc

# =========================
# Marker sets (FB context)
# =========================
markers = {
    "FB_core": ["COL1A1", "DCN", "LUM", "PDGFRA"],
    "ECM_remodeling": ["MMP2", "TIMP1", "TIMP2", "VCAN"],
    "Perivascular_SMC": ["ACTA2", "TAGLN", "MYH11", "PDGFRB"],
    "CM_contamination": ["TNNT2", "MYH6", "MYH7", "NPPA"],
    "Immune_contamination": ["PTPRC", "CD3D", "CD8A", "NKG7", "CD68", "FCGR3A"],
    "Vascular_contamination": ["PECAM1", "VWF", "CD93"]
}

marker_list = []
for genes in markers.values():
    marker_list += present(genes, fb_sub)

# =========================
# 1. Marker Dotplot
# =========================
sc.pl.dotplot(
    fb_sub,
    marker_list,
    groupby="leiden",
    standard_scale="var",
    dot_min=0.05,
    dot_max=0.8,
    cmap="Reds",
    title="FB subset: marker genes (FB / SMC / contamination)"
)

# =========================
# 2. Rank genes
# =========================
sc.tl.rank_genes_groups(
    fb_sub,
    groupby="leiden",
    method="wilcoxon",
    n_genes=10
)

# =========================
# 3. Ranked genes dotplot
# =========================
sc.pl.rank_genes_groups_dotplot(
    fb_sub,
    n_genes=5,
    standard_scale="var",
    cmap="Reds",
    title="FB subset: top ranked genes per cluster",
    dendrogram=False
)

# =========================
# 4. Optional combined view (markers + ranked genes)
# =========================
ranked = fb_sub.uns["rank_genes_groups"]["names"]
top_ranked = list(set(ranked.flatten()))[:30]

combined_genes = marker_list + top_ranked

sc.pl.dotplot(
    fb_sub,
    combined_genes,
    groupby="leiden",
    standard_scale="var",
    dot_min=0.05,
    dot_max=0.8,
    cmap="Reds",
    title="FB subset: markers + ranked genes",
    dendrogram=False
)

In [ ]:
sc.tl.rank_genes_groups(
    fb_sub,
    groupby="leiden",
    method="wilcoxon",
    n_genes=20
)

sc.pl.rank_genes_groups(
    fb_sub,
    sharey=False,
    title="Ranked genes"
)

## Subcluster the recorded fibroblast main clusters

In [ ]:
import scanpy as sc
import pandas as pd

# =========================================================
# Marker sets for fibroblast subclustering
# =========================================================
markers = {
    "FB_core": ["COL1A1", "DCN", "LUM", "PDGFRA"],
    "ECM_remodeling": ["MMP2", "TIMP1", "TIMP2", "VCAN"],
    "Perivascular_SMC": ["ACTA2", "TAGLN", "MYH11", "PDGFRB"],
    "CM_contamination": ["TNNT2", "MYH6", "MYH7", "NPPA"],
    "Immune_contamination": ["PTPRC", "CD3D", "CD8A", "NKG7", "CD68", "FCGR3A"],
    "Vascular_contamination": ["PECAM1", "VWF", "CD93"]
}

def present(genes, adata):
    return [g for g in genes if g in adata.var_names]

def analyze_cluster(adata, cluster_id, resolution=0.5, n_ranked=20):
    print(f"\n==============================")
    print(f"Cluster {cluster_id} | resolution={resolution}")
    print(f"==============================")

    # -------------------------
    # Subset
    # -------------------------
    sub = adata[adata.obs["leiden"].astype(str) == str(cluster_id)].copy()
    print(sub)

    # -------------------------
    # Preprocessing
    # -------------------------
    sc.pp.normalize_total(sub, target_sum=1e4)
    sc.pp.log1p(sub)

    sc.tl.pca(sub, n_comps=30)
    sc.pp.neighbors(sub, n_neighbors=15, n_pcs=30)
    sc.tl.leiden(sub, resolution=resolution, flavor="leidenalg")
    sc.tl.umap(sub)

    # -------------------------
    # Marker list
    # -------------------------
    marker_list = []
    for k in markers:
        marker_list += present(markers[k], sub)

    print("Markers used:", marker_list)

    print("\nSubcluster sizes:")
    print(sub.obs["leiden"].value_counts())

    # -------------------------
    # UMAP
    # -------------------------
    sc.pl.umap(
        sub,
        color=["leiden"],
        title=f"Cluster {cluster_id} subclustering (res={resolution})"
    )

    # -------------------------
    # Marker-Dotplot
    # -------------------------
    sc.pl.dotplot(
        sub,
        marker_list,
        groupby="leiden",
        standard_scale="var",
        dot_min=0.05,
        dot_max=0.8,
        cmap="Reds",
        title=f"Cluster {cluster_id}: FB vs SMC vs contamination",
        dendrogram=False
    )

    # -------------------------
    # Ranked genes nur wenn >1 Subcluster
    # -------------------------
    n_subclusters = sub.obs["leiden"].nunique()

    if n_subclusters >= 2:
        sc.tl.rank_genes_groups(
            sub,
            groupby="leiden",
            method="wilcoxon",
            n_genes=n_ranked
        )

        sc.pl.rank_genes_groups_dotplot(
            sub,
            n_genes=5,
            standard_scale="var",
            cmap="Reds",
            title=f"Cluster {cluster_id}: top genes",
            dendrogram=False
        )

        ranked = sc.get.rank_genes_groups_df(sub, group=None)

        print("\n=== Top ranked genes per subcluster ===")
        for scl in sorted(sub.obs["leiden"].astype(str).unique(), key=lambda x: int(x)):
            top_cl = ranked[ranked["group"] == scl].head(n_ranked)
            print(f"\nSubcluster {scl}")
            print(top_cl[["names", "logfoldchanges", "pvals_adj"]])

    else:
        ranked = None
        print("\nNur ein Subcluster gefunden -> keine ranked genes zwischen Subclustern moeglich.")

    return sub, ranked

In [ ]:
# =========================================================
# Clusters selected for review within the fibroblast compartment
# Cluster 4 is directly reannotated as CM and is therefore not subclustered here
# =========================================================
clusters_to_check = ["10", "14", "11", "17", "16"]

cluster_resolutions = {
    "10": 0.6,
    "14": 0.6,
    "11": 0.6,
    "17": 0.7,
    "16": 0.5,
}

subclusters = {}
ranked_tables = {}

fb_sub.obs["leiden"] = fb_sub.obs["leiden"].astype(str)

for cl in clusters_to_check:
    print(f"\n==============================")
    print(f"Cluster {cl}")
    print(f"==============================")

    res = cluster_resolutions.get(cl, 0.5)

    sub, ranked = analyze_cluster(fb_sub, cl, resolution=res)

    subclusters[cl] = sub
    ranked_tables[cl] = ranked

    subcol = f"leiden_{cl}_sub"

    if subcol not in fb_sub.obs.columns:
        fb_sub.obs[subcol] = pd.Series(index=fb_sub.obs_names, dtype="object")

    fb_sub.obs.loc[sub.obs_names, subcol] = sub.obs["leiden"].astype(str).values

In [ ]:
for cl in ["10", "14", "11", "17", "16"]:
    subcol = f"leiden_{cl}_sub"
    
    if subcol in fb_sub.obs.columns:
        if subcol not in adata.obs.columns:
            adata.obs[subcol] = pd.Series(index=adata.obs_names, dtype="object")
        
        adata.obs.loc[fb_sub.obs_names, subcol] = fb_sub.obs[subcol]

In [ ]:
fb_sub.obs.loc[
    fb_sub.obs["leiden_16_sub"] == "0",
    ["FB_mean", "Immune_mean", "CM_mean", "Vasc_mean"]
].mean()

## Apply the original subcluster mappings and save final annotation

In [ ]:
import pandas as pd

# =========================================================
# 0) Preparation
# =========================================================
fb_sub.obs["leiden"] = fb_sub.obs["leiden"].astype(str)

# Initialize all cells as FB before applying the manual mapping
fb_sub.obs["celltype_FB_refined"] = "FB"

# =========================================================
# 1) Subcluster-Mappings
# =========================================================
fb_maps = {
    "10": {
        "0": "CM_contaminated",
        "1": "vascular",
        "2": "FB",
    },
    "14": {
        "0": "FB",
        "1": "vascular",
        "2": "FB",
        "3": "vascular",
        "4": "CM_contaminated",
    },
    "11": {
        "0": "CM_contaminated",
        "1": "CM_contaminated",
        "2": "vascular",
        "3": "vascular",
    },
    "17": {
        "0": "CM_contaminated",
        "1": "FB",
        "2": "FB",
        "3": "CM_contaminated",
        "4": "immune",
    },
    "16": {
        "0": "FB",
        "1": "CM_contaminated",
        "2": "immune_contaminated",
        "3": "vascular",
    },
}

# =========================================================
# 2) Subcluster-Reannotation anwenden
# =========================================================
for main_clust, mapping in fb_maps.items():
    subcol = f"leiden_{main_clust}_sub"
    mask = fb_sub.obs["leiden"] == main_clust

    if subcol not in fb_sub.obs.columns:
        print(f"Warning: {subcol} not found. Skipping cluster {main_clust}.")
        continue

    fb_sub.obs.loc[mask, "celltype_FB_refined"] = (
        fb_sub.obs.loc[mask, subcol]
        .astype(str)
        .map(mapping)
        .fillna(f"{main_clust}_unassigned")
    )

# =========================================================
# 3) Direkte Hauptcluster-Reannotation
# =========================================================
# Cluster 4 ist komplett CM
fb_sub.obs.loc[fb_sub.obs["leiden"] == "4", "celltype_FB_refined"] = "CM"

# =========================================================
# 4) Kontrolle in fb_sub
# =========================================================
print("=== Hauptcluster vs. FB refined annotation ===")
print(pd.crosstab(fb_sub.obs["leiden"], fb_sub.obs["celltype_FB_refined"]))

print("\n=== Finale Annotationen in fb_sub ===")
print(fb_sub.obs["celltype_FB_refined"].value_counts(dropna=False))

# =========================================================
# 5) Zurück auf adata übertragen
# =========================================================
adata.obs["celltype_final_corrected_v4"] = adata.obs["celltype_final_corrected_v3"].astype(str)

common_cells = adata.obs_names.intersection(fb_sub.obs_names)

adata.obs.loc[common_cells, "celltype_final_corrected_v4"] = (
    fb_sub.obs.loc[common_cells, "celltype_FB_refined"].astype(str)
)

# =========================================================
# 6) Kategorien setzen
# =========================================================
adata.obs["celltype_final_corrected_v4"] = pd.Categorical(
    adata.obs["celltype_final_corrected_v4"],
    categories=[
        "CM",
        "CM_contaminated",
        "FB",
        "vascular",
        "immune",
        "immune_contaminated",
    ],
)

# =========================================================
# 7) Kontrolle in adata
# =========================================================
print("\n=== Verteilung in adata nach FB-Korrektur ===")
print(adata.obs["celltype_final_corrected_v4"].value_counts(dropna=False))

print("\n=== Vergleich vorher vs. nach FB-Korrektur für Zellen aus fb_sub ===")
print(pd.crosstab(
    adata.obs.loc[common_cells, "celltype_final_corrected_v3"],
    adata.obs.loc[common_cells, "celltype_final_corrected_v4"]
))

In [ ]:
markers = {
    "CM": ["TNNT2", "MYH6", "MYH7", "NPPA"],
    "FB": ["COL1A1", "DCN", "LUM", "PDGFRA"],
    "Vascular": ["PECAM1", "VWF", "CD93"],
    "Immune": ["PTPRC", "CD3D", "NKG7"],
}

marker_list = [g for genes in markers.values() for g in genes if g in adata.var_names]

sc.pl.dotplot(
    adata,
    marker_list,
    groupby="celltype_final_corrected_v4",
    standard_scale="var",
    cmap="Reds",
    dot_min=0.05,
    dot_max=0.8,
    title="Final annotation sanity check"
)

In [ ]:
sc.pl.umap(
    adata,
    color="celltype_final_corrected_v4",
    legend_loc="on data"
)

In [ ]:
adata.obs["celltype_final_corrected_v4"].value_counts(normalize=True)

In [ ]:
import numpy as np
import pandas as pd

def present(genes, adata):
    return [g for g in genes if g in adata.var_names]

cm_markers = present(["TNNT2", "MYH6", "MYH7", "NPPA", "NPPB"], adata)
fb_markers = present(["COL1A1", "DCN", "LUM", "PDGFRA"], adata)
vasc_markers = present(["PECAM1", "VWF", "CD93"], adata)
imm_markers = present(["PTPRC", "CD3D", "CD3E", "NKG7"], adata)

adata.obs["CM_mean_global"] = np.array(adata[:, cm_markers].X.mean(axis=1)).flatten()
adata.obs["FB_mean_global"] = np.array(adata[:, fb_markers].X.mean(axis=1)).flatten()
adata.obs["Vasc_mean_global"] = np.array(adata[:, vasc_markers].X.mean(axis=1)).flatten()
adata.obs["Immune_mean_global"] = np.array(adata[:, imm_markers].X.mean(axis=1)).flatten()

adata.obs.groupby("celltype_final_corrected_v4", observed=False)[
    ["CM_mean_global", "FB_mean_global", "Vasc_mean_global", "Immune_mean_global"]
].mean().round(2)

In [ ]:
adata.write_h5ad(OUTPUT_FILE, compression="gzip")
print(f"Saved: {OUTPUT_FILE.resolve()}")
print(adata.obs["celltype_final_corrected_v4"].value_counts(dropna=False))